# Causal discovery with CausalEdu
교육 분야의 중요한 과제 중 하나는 개인별 최적 학습 순서를 설계하는 것입니다.

이를 위해 실제 온라인 교육 플랫폼 Eedi의 로그를 기반으로 구축된 CausalEdu 데이터셋을 사용하여,

Discovery → Identification → Estimation → Refutation의 전 과정을 PyWhy 스택으로 구현합니다.

In [1]:
%pip -q install lingam

Note: you may need to restart the kernel to use updated packages.


In [39]:
import warnings, numpy as np, pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
SEED = 18
np.random.seed(SEED)
plt.style.use("default")

## Data Setup
CausalEdu는 Eedi 플랫폼에서 수집된 관찰 데이터 및 일부 학습 개념 쌍(construct pairs)에 대해 A/B 테스트 결과와 전문가 지식이 포함된 데이터셋입니다.

- **checkins_lessons_checkouts_training.csv**: 관찰 데이터
  - `ConstructId`는 학습 개념(예: ‘분수 덧셈’)을 나타냅니다.
  - 한 세션은 Check-in → Lesson → Check-out 순서로 진행됩니다.

- **checkin_to_checkout.csv**: A/B 테스트 결과(Discovery용 Ground Truth)
    - LessonConstructId → QuestionConstructId 쌍에 대해 학습 전·후 정답 패턴(`n00, n01, n10, n11`)을 제공합니다.
    - 두 가지 가설로부터 **p010_m**, **k010_m** 통계를 제공합니다.

        - 가설 1(학습은 해롭지 않다): $p010\_m = \frac{n_{01}}{n_{00}+n_{01}}$
        - 가설 2(우연 정답 보정): $k010\_m = \frac{n_{01}}{n_{00}+n_{01}+n_{10}}$

- **construct_prerequisites_test.csv**: 전문가 지식  
   - 전문가가 정의한 개념 간 선행관계(prerequisite structure)를 제공합니다.

- **construct_experiments_ates_test.csv**: A/B 테스트 결과(Inference용 Ground Truth)
  - A/B 테스트을 통해 측정된 CATE로 특정 레슨 개념(`TreatmentLessonConstructId`)이 다른 개념(`QuestionConstructId`)에 미치는 실제 개입 효과를 제공합니다.

In [47]:
DATA_DIR = Path("../data/causal_edu")

training = pd.read_csv(DATA_DIR / "checkins_lessons_checkouts_training.csv")
c2c = pd.read_csv(DATA_DIR / "checkin_to_checkout.csv")
expert = pd.read_csv(DATA_DIR / "construct_prerequisites_test.csv")
ab_ates = pd.read_csv(DATA_DIR / "construct_experiments_ates_test.csv")
q_input = pd.read_csv(DATA_DIR / "construct_experiments_input_test.csv")

print("training:", training.shape)
display(training.head(3))
print("c2c:", c2c.shape)
display(c2c.head(3))
print("expert:", expert.shape)
display(expert.head(3))
print("ab_ates:", ab_ates.shape)
display(ab_ates.head(3))
print("q_input:", q_input.shape)
display(q_input.head(3))


training: (641490, 12)


,QuizSessionId,AnswerId,UserId,QuizId,QuestionId,IsCorrect,AnswerValue,CorrectAnswer,QuestionSequence,ConstructId,Type,Timestamp
0,0,0.0,0,242762,130326,1.0,4.0,4.0,1,9,Checkin,2022-02-01 01:53:31.170
1,1,1.0,1,242762,130326,1.0,4.0,4.0,1,9,Checkin,2022-02-01 02:07:31.393
2,1,2.0,1,242762,130327,0.0,1.0,4.0,2,10,Checkin,2022-02-01 02:08:27.947


c2c: (183, 9)


,LessonConstructId,QuestionConstructId,n00,n01,n10,n11,Count,p010_m,k010_m
0,70,1270,8,38,3,30,79,0.826087,0.775510
1,70,1271,56,5,6,11,78,0.081967,0.074627
2,70,1272,13,10,18,36,77,0.434783,0.243902


expert: (3277, 3)


,ConstructId,SubjectId,PrerequisiteConstructIds
0,854,33.0,{76}
1,855,33.0,{76}
2,856,33.0,"{483, 76}"


ab_ates: (88, 8)


,TreatmentLessonConstructId,QuestionConstructId,Year,ControlLessonConstructIds,ControlUsersCount,TreatmentUsersCount,ate_p_1__,ate_k_1__
0,206,211,7,{3119},73,94,0.033656,-0.019091
1,206,212,7,{3119},77,101,-0.022222,-0.036004
2,206,216,7,{3119},75,94,-0.109501,-0.118014


q_input: (45, 4)


,TreatmentLessonConstructId,QuestionConstructId,Year,ControlLessonConstructIds
0,206,211,7,{3119}
1,206,212,7,{3119}
2,206,216,7,{3119}


### 데이터 변환 (pivot)

세션 로그를 (UserId, QuizSessionId) × ConstructId 형태의 행렬로 변환해  
Discovery 구조학습 알고리즘(PC/GES/NOTEARS)이 사용할 수 있는 입력 형태로 만듭니다.

변환 과정은 다음과 같습니다:
1. `training`에서 Checkout/CheckoutRetry만 사용하여 Construct에 대한 최종 결과에 집중
2. 동일한 (User, Session, Construct) 내 마지막 시도만 선택
3. 피벗하여 (UserId, QuizSessionId)를 row index, ConstructId를 column으로 구성
4. 결측은 0으로 채우고, 시도 여부는 별도 mask로 관리

In [48]:
df = training.sort_values(["UserId", "QuizSessionId", "Timestamp"])
checkout = df[df["Type"].isin(["Checkout", "CheckoutRetry"])]
print("checkout:", checkout.shape)
display(checkout.head())

checkout: (78373, 12)


,QuizSessionId,AnswerId,UserId,QuizId,QuestionId,IsCorrect,AnswerValue,CorrectAnswer,QuestionSequence,ConstructId,Type,Timestamp
24,4,23.0,1,226964,129234,0.0,1.0,4.0,3,2480,Checkout,2022-02-01 02:23:32.213
25,4,24.0,1,226964,129234,1.0,4.0,4.0,3,2480,CheckoutRetry,2022-02-01 02:24:01.030
4231,527,3900.0,1,228600,130029,0.0,2.0,1.0,1,48,Checkout,2022-02-02 04:40:27.407
4232,527,3901.0,1,228600,130029,1.0,1.0,1.0,1,48,CheckoutRetry,2022-02-02 04:40:48.850
32,5,30.0,2,202086,104952,1.0,2.0,2.0,1,1271,Checkout,2022-02-01 04:18:31.917


In [49]:
last_checkout = (
    checkout.sort_values("Timestamp")
            .groupby(["UserId", "QuizSessionId", "ConstructId"], as_index=False)
            .tail(1)
)
print("last_checkout:", last_checkout.shape)
display(last_checkout.head())

last_checkout: (47677, 12)


,QuizSessionId,AnswerId,UserId,QuizId,QuestionId,IsCorrect,AnswerValue,CorrectAnswer,QuestionSequence,ConstructId,Type,Timestamp
25,4,24.0,1,226964,129234,1.0,4.0,4.0,3,2480,CheckoutRetry,2022-02-01 02:24:01.030
32,5,30.0,2,202086,104952,1.0,2.0,2.0,1,1271,Checkout,2022-02-01 04:18:31.917
46,6,42.0,3,202371,120911,1.0,1.0,1.0,1,1815,Checkout,2022-02-01 04:21:57.063
37,5,34.0,2,202086,104953,1.0,2.0,2.0,2,3133,CheckoutRetry,2022-02-01 04:23:00.313
57,7,52.0,4,202465,76153,0.0,3.0,1.0,3,2257,CheckoutRetry,2022-02-01 04:31:45.893


In [50]:
wide_z = last_checkout.pivot_table(
    index=["UserId", "QuizSessionId"],
    columns="ConstructId",
    values="IsCorrect",
    aggfunc="max"
).fillna(0).astype(float)

print("wide_z shape:", wide_z.shape)
display(wide_z.head(3))

wide_z shape: (35517, 1063)


ConstructId           4     9     10    20    22    26    27    28    29    \
UserId QuizSessionId                                                         
1      4               0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   
       527             0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   
2      5               0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   

ConstructId           30    ...  3442  3443  3450  3451  3487  3489  3510  \
UserId QuizSessionId        ...                                             
1      4               0.0  ...   0.0   0.0   0.0   0.0   0.0   0.0   0.0   
       527             0.0  ...   0.0   0.0   0.0   0.0   0.0   0.0   0.0   
2      5               0.0  ...   0.0   0.0   0.0   0.0   0.0   0.0   0.0   

ConstructId           3511  3513  3515  
UserId QuizSessionId                    
1      4               0.0   0.0   0.0  
       527             0.0   0.0   0.0  
2      5               0.0   0.0   0.0  

[3 rows x 1063 columns]

In [51]:
# attempt mask 생성: '시도 여부'만 1, 나머지 0
attempt_flag = last_checkout.assign(_attempt=1).pivot_table(
    index=["UserId","QuizSessionId"],
    columns="ConstructId",
    values="_attempt",
    aggfunc="max"
).fillna(0).astype(int)

# 각 컬럼 시도율(=시도한 세션 비율) 상위 5개 확인
print("attempt_flag shape:", attempt_flag.shape)
attempt_rate = attempt_flag.mean(axis=0).rename("attempt_rate").sort_values(ascending=False)
display(attempt_rate.head().to_frame().T)


attempt_flag shape: (35517, 1063)


ConstructId,331,2908,3316,342,3343
attempt_rate,0.010361,0.00977,0.009545,0.009348,0.008306


In [52]:
# 모든 세션에서 단 한 번도 등장하지 않은 개념(Construct) 제거
#     → 학생들이 전체 기간 동안 시도조차 하지 않은 개념
allzero_cols = [c for c in wide_z.columns if (wide_z[c] == 0).all()]
print("all-zero columns:", len(allzero_cols))

wide_z_nz   = wide_z.drop(columns=allzero_cols)   if allzero_cols else wide_z
attempt_nz  = attempt_flag.drop(columns=allzero_cols) if allzero_cols else attempt_flag


all-zero columns: 7


시도율이 0.1% 미만인 희소 개념은 Discovery 안정성이 떨어진다고 판단해 제거합니다.
(대략 35k 세션 기준, 40회 미만 등장하는 개념)

In [93]:
# 시도율 적은 construct 제거
thr = 0.001  # 0.1%
keep_cols = [c for c, r in attempt_rate.items() if r >= thr]

wide_z_keep  = wide_z_nz[keep_cols]
attempt_keep = attempt_nz[keep_cols]

print("kept constructs:", len(keep_cols))
print("wide_z_keep:", wide_z_keep.shape)

kept constructs: 460
wide_z_keep: (35517, 460)


## Causal Discovery

`wide_z_keep`을 입력으로 인과 그래프를 학습하고,  
A/B 테스트와 전문가 지식를 이용해 Discovery 성능을 평가합니다.  

핵심은 **학습에 사용하는 BK**와 **평가에 사용하는 Ground Truth**를 분리하는 것입니다.

### 학습 데이터 (with Background Knowledge)

- `wide_z_keep`  
- **Temporal BK**  
  - 시간·퀴즈 순서를 거스르는 edge 금지
- `expert_bk_edges`  
  - Expert edge 중 학습용(80%)만 사용  
  - require / soft prior로 사용

### 평가 데이터 (Ground Truth)
- `c2c_eval_edges`
    - 학습에는 사용하지 않음  
    - Discovery 평가(F1 / Precision / Recall / SHD)에만 사용
- `expert_holdout_edges` (~20%)
    - 학습에는 절대 포함하지 않음  
    - Discovery 평가용 GT로만 사용

> C2C는 A/B 실험 기반이므로 Expert와 충돌 시 **C2C 우선**.

In [94]:
# Discovery에서 실제 학습에 사용하는 노드 = wide_z_keep 컬럼
training_nodes = set(pd.to_numeric(pd.Index(wide_z_keep.columns), errors="coerce").dropna().astype(int))

print("Training nodes (wide_z_keep):", len(training_nodes))
print(training_nodes)

Training nodes (wide_z_keep): 460
{2052, 9, 10, 28, 29, 2077, 31, 30, 36, 39, 2087, 42, 45, 46, 47, 48, 49, 50, 2105, 60, 2113, 65, 70, 71, 73, 74, 2126, 84, 2141, 2142, 95, 2144, 100, 2151, 107, 2161, 2165, 161, 2213, 172, 174, 180, 2229, 192, 196, 197, 198, 203, 2252, 2253, 206, 207, 208, 2257, 210, 211, 209, 205, 216, 217, 218, 220, 221, 223, 224, 230, 234, 2282, 237, 2286, 246, 2295, 2296, 248, 2298, 2297, 2299, 256, 261, 265, 270, 274, 277, 284, 287, 296, 297, 298, 299, 301, 302, 306, 2357, 311, 312, 313, 314, 315, 318, 321, 325, 328, 331, 334, 336, 338, 342, 2391, 343, 345, 2393, 346, 352, 353, 354, 356, 2406, 358, 362, 363, 366, 368, 369, 371, 2420, 373, 376, 378, 380, 381, 2429, 383, 384, 389, 390, 396, 397, 399, 400, 401, 402, 403, 2451, 408, 409, 411, 414, 415, 2462, 419, 422, 423, 425, 2480, 433, 435, 437, 440, 441, 442, 444, 446, 447, 448, 457, 458, 464, 465, 466, 467, 470, 471, 473, 479, 483, 2548, 507, 2560, 2561, 2562, 2563, 2564, 2566, 2570, 2571, 2574, 2576, 2578, 531,

In [95]:
# C2C edge 테이블 (Lesson → Question)
c2c_edges_all = (
    c2c.assign(
        u=pd.to_numeric(c2c["LessonConstructId"], errors="coerce"),
        v=pd.to_numeric(c2c["QuestionConstructId"], errors="coerce"),
    )
    .dropna(subset=["u", "v"])
)

c2c_edges_all["u"] = c2c_edges_all["u"].astype(int)
c2c_edges_all["v"] = c2c_edges_all["v"].astype(int)

print("c2c_edges_all:", c2c_edges_all.shape)
display(c2c_edges_all.head())


c2c_edges_all: (183, 11)


,LessonConstructId,QuestionConstructId,n00,n01,n10,n11,Count,p010_m,k010_m,u,v
0,70,1270,8,38,3,30,79,0.826087,0.775510,70,1270
1,70,1271,56,5,6,11,78,0.081967,0.074627,70,1271
2,70,1272,13,10,18,36,77,0.434783,0.243902,70,1272
3,70,1403,21,9,22,24,76,0.300000,0.173077,70,1403
4,70,3380,21,4,10,41,76,0.160000,0.114286,70,3380


In [96]:
# 평가용 C2C: u 또는 v 중 하나라도 training_nodes에 걸리는 edge만 사용
mask_any = (
    c2c_edges_all["u"].isin(training_nodes)
    & c2c_edges_all["v"].isin(training_nodes)
)

c2c_eval_edges = c2c_edges_all.loc[mask_any, ["u", "v", "p010_m", "k010_m", "Count"]].drop_duplicates()

print("c2c_eval_edges (GT용):", c2c_eval_edges.shape)
display(c2c_eval_edges.head())

c2c_eval_edges (GT용): (69, 5)


,u,v,p010_m,k010_m,Count
1,70,1271,0.081967,0.074627,78
3,70,1403,0.300000,0.173077,76
5,206,211,0.424242,0.245614,94
7,206,216,0.182927,0.129630,214
8,206,217,0.576271,0.400407,258


In [97]:
# Expert edge 테이블: u → v
rows = []
for _, row in expert.iterrows():
    v = pd.to_numeric(row["ConstructId"], errors="coerce")
    if pd.isna(v):
        continue
    v = int(v)

    prereq_raw = str(row["PrerequisiteConstructIds"]) if pd.notna(row["PrerequisiteConstructIds"]) else ""
    tokens = prereq_raw.replace(",", "|").split("|")
    for t in tokens:
        t = t.strip()
        if not t:
            continue
        u = pd.to_numeric(t, errors="coerce")
        if pd.isna(u):
            continue
        u = int(u)
        rows.append((u, v))

expert_edges_all = pd.DataFrame(rows, columns=["u", "v"]).drop_duplicates()
print("expert_edges_all:", expert_edges_all.shape)
display(expert_edges_all.head())


expert_edges_all: (1391, 2)


,u,v
0,480,857
1,754,756
2,755,758
3,755,760
4,761,763


In [98]:
# wide_z_keep 에 등장하는 노드만 사용
expert_edges_train = expert_edges_all[
    expert_edges_all["u"].isin(training_nodes)
    & expert_edges_all["v"].isin(training_nodes)
].reset_index(drop=True)

print("expert_edges_train:", expert_edges_train.shape)
display(expert_edges_train.head())


expert_edges_train: (96, 2)


,u,v
0,331,2930
1,342,2930
2,483,458
3,302,464
4,3347,464


In [99]:
# 80:20 split → expert_bk_edges / expert_holdout_edges
frac_bk = 0.8
expert_edges_shuffled = expert_edges_train.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

n_bk = int(len(expert_edges_shuffled) * frac_bk)

expert_bk_edges     = expert_edges_shuffled.iloc[:n_bk].reset_index(drop=True)
expert_holdout_edges = expert_edges_shuffled.iloc[n_bk:].reset_index(drop=True)

print("expert_bk_edges     (BK) :", expert_bk_edges.shape)
print("expert_holdout_edges(GT) :", expert_holdout_edges.shape)


expert_bk_edges     (BK) : (76, 2)
expert_holdout_edges(GT) : (20, 2)
